In [1]:
!pip install -q anthropic openpyxl pandas numpy requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 11.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.chart import BarChart, LineChart, Reference
import anthropic
import json
import os
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings("ignore")
from openpyxl.chart.label import DataLabelList

print("✅ All libraries loaded successfully!\n")

✅ All libraries loaded successfully!



In [3]:
def generate_retail_data():
    """
    Generates 8 weeks of retail transaction data across 5 stores.
    Think of this as your SQL export from the company's database.

    Real-world equivalent:
        df = pd.read_sql("SELECT * FROM sales WHERE date >= '2024-01-01'", conn)
    """
    print("📦 STEP 1: Generating retail dataset...")
    print("   (In real jobs: this is your SQL/CSV import)\n")

    random.seed(42)
    np.random.seed(42)

    # --- Master Data (like dimension tables in a data warehouse) ---
    stores = {
        "S001": {"name": "Mumbai Central",   "region": "West",  "tier": "Premium"},
        "S002": {"name": "Delhi Connaught",  "region": "North", "tier": "Premium"},
        "S003": {"name": "Bangalore MG Rd",  "region": "South", "tier": "Standard"},
        "S004": {"name": "Chennai T Nagar",  "region": "South", "tier": "Standard"},
        "S005": {"name": "Pune FC Road",     "region": "West",  "tier": "Budget"},
    }

    categories = ["Electronics", "Clothing", "Groceries", "Home & Living", "Sports"]

    # --- Generate 8 weeks of daily transactions ---
    records = []
    start_date = datetime(2024, 11, 4)  # Week 1 start

    for week_num in range(1, 9):
        week_start = start_date + timedelta(weeks=week_num - 1)

        for store_id, store_info in stores.items():
            for day in range(7):  # 7 days per week
                txn_date = week_start + timedelta(days=day)

                # Simulate realistic business patterns
                is_weekend = txn_date.weekday() >= 5
                weekend_boost = 1.4 if is_weekend else 1.0

                # Inject anomaly: Store S003 crashes in Week 6 (we'll detect this!)
                anomaly_factor = 0.35 if (store_id == "S003" and week_num == 6) else 1.0

                # Inject anomaly: S001 spikes in Week 8 (festival season)
                spike_factor = 1.8 if (store_id == "S001" and week_num == 8) else 1.0

                for category in categories:
                    # Base revenue varies by store tier
                    base = {"Premium": 85000, "Standard": 55000, "Budget": 30000}[store_info["tier"]]
                    category_multiplier = {"Electronics": 1.5, "Clothing": 1.2,
                                           "Groceries": 0.8, "Home & Living": 1.0, "Sports": 0.9}[category]

                    revenue = (base * category_multiplier * weekend_boost *
                               anomaly_factor * spike_factor *
                               np.random.uniform(0.85, 1.15))

                    units_sold = int(revenue / np.random.uniform(450, 850))
                    returns = int(units_sold * np.random.uniform(0.02, 0.08))
                    cogs = revenue * np.random.uniform(0.55, 0.68)

                    records.append({
                        "date":          txn_date.strftime("%Y-%m-%d"),
                        "week_number":   week_num,
                        "store_id":      store_id,
                        "store_name":    store_info["name"],
                        "region":        store_info["region"],
                        "tier":          store_info["tier"],
                        "category":      category,
                        "revenue":       round(revenue, 2),
                        "units_sold":    units_sold,
                        "returns":       returns,
                        "cogs":          round(cogs, 2),
                        "footfall":      int(units_sold * np.random.uniform(3.5, 6.0)),
                        "discount_pct":  round(np.random.uniform(5, 25), 1),
                    })

    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["date"])
    df["gross_profit"] = df["revenue"] - df["cogs"]
    df["net_units"] = df["units_sold"] - df["returns"]
    df["return_rate"] = (df["returns"] / df["units_sold"] * 100).round(2)
    df["conversion_rate"] = (df["net_units"] / df["footfall"] * 100).round(2)
    df["profit_margin"] = (df["gross_profit"] / df["revenue"] * 100).round(2)

    print(f"   ✅ Dataset created: {len(df):,} rows × {len(df.columns)} columns")
    print(f"   📅 Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"   🏪 Stores: {df['store_name'].nunique()} | Categories: {df['category'].nunique()}\n")
    return df



In [4]:
def compute_kpis(df):
    """
    Computes business KPIs — things your manager will ask about in meetings.

    KPIs computed:
    - Total Revenue & WoW Growth %
    - Gross Profit & Profit Margin
    - Units Sold, Return Rate
    - Footfall & Conversion Rate
    - Best/Worst performing store & category
    """
    print("📊 STEP 2: Computing KPIs (replacing Monday morning Excel work)...\n")

    # --- KPI 1: Weekly Summary ---
    weekly = df.groupby("week_number").agg(
        total_revenue    = ("revenue",         "sum"),
        total_profit     = ("gross_profit",    "sum"),
        total_units      = ("net_units",       "sum"),
        total_returns    = ("returns",         "sum"),
        total_footfall   = ("footfall",        "sum"),
        avg_margin       = ("profit_margin",   "mean"),
        avg_return_rate  = ("return_rate",     "mean"),
        avg_conversion   = ("conversion_rate", "mean"),
    ).reset_index()

    # Week-over-Week growth (the metric every retail analyst tracks)
    weekly["wow_growth_pct"] = weekly["total_revenue"].pct_change() * 100
    weekly["wow_growth_pct"] = weekly["wow_growth_pct"].round(2)
    weekly["total_revenue"]  = weekly["total_revenue"].round(2)
    weekly["total_profit"]   = weekly["total_profit"].round(2)
    weekly["avg_margin"]     = weekly["avg_margin"].round(2)

    print("   📈 Weekly Revenue Summary:")
    for _, row in weekly.iterrows():
        arrow = "▲" if (row["wow_growth_pct"] > 0 or pd.isna(row["wow_growth_pct"])) else "▼"
        growth_str = f"{arrow} {abs(row['wow_growth_pct']):.1f}%" if not pd.isna(row["wow_growth_pct"]) else "  —  "
        print(f"      Week {int(row['week_number'])}: ₹{row['total_revenue']:>12,.0f}  |  WoW: {growth_str}")

    # --- KPI 2: Store Performance (last 2 weeks) ---
    recent_weeks = df[df["week_number"] >= 7]
    store_kpi = recent_weeks.groupby(["store_id", "store_name", "tier"]).agg(
        revenue      = ("revenue",         "sum"),
        profit       = ("gross_profit",    "sum"),
        units        = ("net_units",       "sum"),
        margin       = ("profit_margin",   "mean"),
        conversion   = ("conversion_rate", "mean"),
        return_rate  = ("return_rate",     "mean"),
    ).reset_index()
    store_kpi["margin"]      = store_kpi["margin"].round(2)
    store_kpi["conversion"]  = store_kpi["conversion"].round(2)
    store_kpi["return_rate"] = store_kpi["return_rate"].round(2)
    store_kpi = store_kpi.sort_values("revenue", ascending=False)

    print(f"\n   🏪 Store Performance (Weeks 7-8):")
    for _, row in store_kpi.iterrows():
        print(f"      {row['store_name']:22s} | Rev: ₹{row['revenue']:>10,.0f} | Margin: {row['margin']:.1f}%")

    # --- KPI 3: Category Performance ---
    cat_kpi = df.groupby("category").agg(
        revenue    = ("revenue",         "sum"),
        profit     = ("gross_profit",    "sum"),
        units      = ("net_units",       "sum"),
        margin     = ("profit_margin",   "mean"),
    ).reset_index().sort_values("revenue", ascending=False)
    cat_kpi["margin"] = cat_kpi["margin"].round(2)

    # --- KPI 4: Region Performance ---
    region_kpi = df.groupby("region").agg(
        revenue = ("revenue",      "sum"),
        profit  = ("gross_profit", "sum"),
        margin  = ("profit_margin","mean"),
    ).reset_index().sort_values("revenue", ascending=False)
    region_kpi["margin"] = region_kpi["margin"].round(2)

    print(f"\n   ✅ KPIs computed successfully!\n")

    return {
        "weekly":     weekly,
        "store_kpi":  store_kpi,
        "cat_kpi":    cat_kpi,
        "region_kpi": region_kpi,
    }


In [5]:
def detect_anomalies(df, kpis):
    """
    Detects business anomalies using statistical rules.

    Anomalies detected:
    1. Revenue drops > 40% WoW for any store
    2. Stores significantly below chain average
    3. Revenue spikes > 50% WoW (investigate — fraud? Data error? Festival?)
    4. High return rates (customer satisfaction issue)
    5. Low conversion rates (footfall not converting to sales)
    """
    print("🔍 STEP 3: Running Anomaly Detection Engine...\n")

    anomalies = []

    # --- Anomaly Type 1: Store-Week Revenue Drops ---
    store_weekly = df.groupby(["store_id", "store_name", "week_number"])["revenue"].sum().reset_index()
    store_weekly = store_weekly.sort_values(["store_id", "week_number"])
    store_weekly["prev_week_rev"] = store_weekly.groupby("store_id")["revenue"].shift(1)
    store_weekly["wow_change"] = ((store_weekly["revenue"] - store_weekly["prev_week_rev"])
                                   / store_weekly["prev_week_rev"] * 100)

    for _, row in store_weekly.iterrows():
        if pd.notna(row["wow_change"]):
            if row["wow_change"] <= -40:
                anomalies.append({
                    "type":        "🚨 CRITICAL DROP",
                    "store":       row["store_name"],
                    "week":        int(row["week_number"]),
                    "detail":      f"Revenue dropped {row['wow_change']:.1f}% vs last week",
                    "value":       f"₹{row['revenue']:,.0f}",
                    "prev_value":  f"₹{row['prev_week_rev']:,.0f}",
                    "priority":    1,
                })
            elif row["wow_change"] >= 60:
                anomalies.append({
                    "type":       "⚠️ REVENUE SPIKE",
                    "store":      row["store_name"],
                    "week":       int(row["week_number"]),
                    "detail":     f"Revenue surged +{row['wow_change']:.1f}% — verify source",
                    "value":      f"₹{row['revenue']:,.0f}",
                    "prev_value": f"₹{row['prev_week_rev']:,.0f}",
                    "priority":   2,
                })

    # --- Anomaly Type 2: Stores Below Chain Average ---
    chain_avg_rev = kpis["store_kpi"]["revenue"].mean()
    for _, row in kpis["store_kpi"].iterrows():
        if row["revenue"] < chain_avg_rev * 0.6:
            anomalies.append({
                "type":        "📉 UNDERPERFORMER",
                "store":       row["store_name"],
                "week":        "7-8",
                "detail":      f"Revenue {((row['revenue']/chain_avg_rev)-1)*100:.1f}% below chain average",
                "value":       f"₹{row['revenue']:,.0f}",
                "prev_value":  f"₹{chain_avg_rev:,.0f} (avg)",
                "priority":    2,
            })

    # --- Anomaly Type 3: High Return Rates ---
    store_returns = df.groupby(["store_name"])["return_rate"].mean()
    chain_return_avg = store_returns.mean()
    for store, rate in store_returns.items():
        if rate > chain_return_avg * 1.5:
            anomalies.append({
                "type":        "🔄 HIGH RETURNS",
                "store":       store,
                "week":        "All",
                "detail":      f"Return rate {rate:.1f}% vs chain avg {chain_return_avg:.1f}%",
                "value":       f"{rate:.1f}%",
                "prev_value":  f"{chain_return_avg:.1f}% (avg)",
                "priority":    3,
            })

    # Sort by priority
    anomalies.sort(key=lambda x: x["priority"])

    print(f"   🚨 Anomalies detected: {len(anomalies)}")
    for a in anomalies:
        print(f"      {a['type']} | {a['store']} | Week {a['week']} | {a['detail']}")

    print()
    return anomalies


In [ ]:
import requests

def generate_ai_report(kpis, anomalies, df):
    print("🤖 STEP 4: Calling Groq AI to generate executive report...\n")

    # ── GET FREE KEY: https://console.groq.com → API Keys → Create ──
    GROQ_API_KEY = ""
    # ────────────────────────────────────────────────────────────────

    weekly       = kpis["weekly"]
    best_store   = kpis["store_kpi"].iloc[0]
    worst_store  = kpis["store_kpi"].iloc[-1]
    best_cat     = kpis["cat_kpi"].iloc[0]
    total_rev    = weekly["total_revenue"].sum()
    total_profit = weekly["total_profit"].sum()
    last_wow     = weekly["wow_growth_pct"].iloc[-1]
    ovr_margin   = round(total_profit / total_rev * 100, 1)

    top_anomalies = "\n".join([
        f"- {a['type']} | {a['store']} Week {a['week']}: {a['detail']}"
        for a in anomalies[:3]
    ])

    prompt = f"""You are a senior retail BI analyst. Write an executive report for C-suite.

DATA:
- 8-Week Revenue: Rs.{total_rev:,.0f} | Profit: Rs.{total_profit:,.0f} | Margin: {ovr_margin}%
- Last Week Growth: {last_wow:+.1f}%
- Best Store: {best_store['store_name']} Rs.{best_store['revenue']:,.0f}
- Worst Store: {worst_store['store_name']} Rs.{worst_store['revenue']:,.0f}
- Top Category: {best_cat['category']} Rs.{best_cat['revenue']:,.0f}
- Anomalies:
{top_anomalies}

Write these 4 sections (2-3 sentences each):
1. EXECUTIVE SUMMARY
2. ANOMALY ANALYSIS
3. RECOMMENDED ACTIONS (3 bullet points)
4. OUTLOOK

Use Rs. for currency. Be specific with numbers."""

    response = requests.post(
        url="https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": "llama-3.3-70b-versatile",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 1024,
            "temperature": 0.7
        },
        timeout=60
    )

    if response.status_code != 200:
        raise Exception(f"Groq error {response.status_code}: {response.text[:300]}")

    report_text = response.json()["choices"][0]["message"]["content"]

    print("   ✅ AI Report generated successfully!")
    print(f"   📝 Words: {len(report_text.split())}\n")
    print("-" * 60)
    print(report_text)
    print("-" * 60 + "\n")
    return report_text

In [11]:
def export_to_excel(df, kpis, anomalies, ai_report):
    print("📁 STEP 5: Building professional Excel report + Dashboard...\n")

    import os
    os.makedirs("/content/retailpulse_outputs", exist_ok=True)

    from openpyxl.chart import BarChart, LineChart, PieChart, Reference
    from openpyxl.chart.series import DataPoint
    from openpyxl.chart.label import DataLabelList

    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    filename  = f"/content/retailpulse_outputs/RetailPulse_{timestamp}.xlsx"

    NAVY="1B2A4A"; TEAL="0D7377"; RED="C0392B"; GREEN="27AE60"
    ORANGE="E67E22"; PURPLE="8E44AD"; LIGHT_BG="F0F4F8"
    LIGHT_TEAL="E8F8F8"; WHITE="FFFFFF"; DARK_TEXT="1A1A2E"

    thin = Side(style="thin", color="CCCCCC")
    def bdr(): return Border(left=thin, right=thin, top=thin, bottom=thin)
    def fill(c): return PatternFill("solid", fgColor=c)

    # ── Write data sheets ──────────────────────────────────────────
    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        df_export = df.copy()
        df_export["date"] = pd.to_datetime(df_export["date"]).dt.strftime("%Y-%m-%d")
        df_export.to_excel(writer,         sheet_name="Raw Data",  index=False)
        kpis["weekly"].to_excel(writer,    sheet_name="wk_data",   index=False)
        kpis["store_kpi"].to_excel(writer, sheet_name="st_data",   index=False)
        kpis["cat_kpi"].to_excel(writer,   sheet_name="cat_data",  index=False)
        if anomalies:
            pd.DataFrame(anomalies).to_excel(writer, sheet_name="Anomalies", index=False)

    wb = openpyxl.load_workbook(filename)

    # ══════════════════════════════════════════════════════════════
    # DASHBOARD SHEET
    # ══════════════════════════════════════════════════════════════
    ws = wb.create_sheet("📊 Dashboard", 0)
    ws.sheet_view.showGridLines = False
    ws.sheet_view.zoomScale = 85

    for col, w in {"A":1,"B":18,"C":18,"D":18,"E":18,"F":18,
                   "G":1,"H":18,"I":18,"J":18,"K":18,"L":18,"M":1}.items():
        ws.column_dimensions[col].width = w

    ws.row_dimensions[1].height = 6

    # Title — value BEFORE merge
    ws.cell(2, 2, "RetailPulse AI  —  Executive KPI Dashboard")
    ws["B2"].font      = Font(bold=True, size=22, color=WHITE, name="Calibri")
    ws["B2"].fill      = fill(NAVY)
    ws["B2"].alignment = Alignment(horizontal="center", vertical="center")
    ws.merge_cells("B2:L3")
    ws.row_dimensions[2].height = 26
    ws.row_dimensions[3].height = 22

    # Subtitle — value BEFORE merge
    ws.cell(4, 2, f"8-Week Performance  ·  {len(df):,} Transactions  ·  5 Stores  ·  5 Categories  ·  {datetime.now().strftime('%d %b %Y')}")
    ws["B4"].font      = Font(italic=True, size=10, color="AAAAAA", name="Calibri")
    ws["B4"].fill      = fill("0F1A30")
    ws["B4"].alignment = Alignment(horizontal="center", vertical="center")
    ws.merge_cells("B4:L4")
    ws.row_dimensions[4].height = 16
    ws.row_dimensions[5].height = 8

    # ── KPI Cards rows 6-8 ────────────────────────────────────────
    weekly      = kpis["weekly"]
    total_rev   = weekly["total_revenue"].sum()
    total_prof  = weekly["total_profit"].sum()
    avg_margin  = weekly["avg_margin"].mean()
    last_wow    = weekly["wow_growth_pct"].iloc[-1]
    avg_conv    = weekly["avg_conversion"].mean()
    avg_return  = weekly["avg_return_rate"].mean()
    total_units = int(weekly["total_units"].sum())
    wow_color   = GREEN if last_wow > 0 else RED

    cards = [
        ("B", "TOTAL REVENUE",   f"Rs.{total_rev/1e7:.1f} Cr",  "8-Week Chain Total",  NAVY),
        ("C", "GROSS PROFIT",    f"Rs.{total_prof/1e7:.1f} Cr", "8-Week Total",         TEAL),
        ("D", "AVG MARGIN",      f"{avg_margin:.1f}%",           "All stores",           "0A6E72"),
        ("E", "WoW GROWTH",      f"{last_wow:+.1f}%",            "Week 7 to Week 8",     wow_color),
        ("H", "UNITS SOLD",      f"{total_units:,}",             "Net of returns",       PURPLE),
        ("I", "CONVERSION RATE", f"{avg_conv:.1f}%",             "Footfall to Purchase", ORANGE),
        ("J", "RETURN RATE",     f"{avg_return:.1f}%",           "Chain average",        RED),
        ("K", "ANOMALIES",       f"{len(anomalies)}",            "Auto-detected",        "6C3483"),
    ]
    ws.row_dimensions[6].height = 16
    ws.row_dimensions[7].height = 30
    ws.row_dimensions[8].height = 14

    for col_letter, label, value, sub, bg in cards:
        col = openpyxl.utils.column_index_from_string(col_letter)
        for r in [6, 7, 8]:
            ws.cell(r, col).fill = fill(bg)
        ws.cell(6, col, label).font      = Font(bold=True, color=WHITE, size=8, name="Calibri")
        ws.cell(6, col).fill             = fill(bg)
        ws.cell(6, col).alignment        = Alignment(horizontal="center", vertical="center")
        ws.cell(7, col, value).font      = Font(bold=True, color=WHITE, size=16, name="Calibri")
        ws.cell(7, col).fill             = fill(bg)
        ws.cell(7, col).alignment        = Alignment(horizontal="center", vertical="center")
        ws.cell(8, col, sub).font        = Font(color=WHITE, size=7, name="Calibri", italic=True)
        ws.cell(8, col).fill             = fill(bg)
        ws.cell(8, col).alignment        = Alignment(horizontal="center", vertical="center")

    ws.row_dimensions[9].height = 8

    # ── Chart data at row 55+ (off-screen) ───────────────────────
    DR = 62
    ws.cell(DR, 1, "Week"); ws.cell(DR, 2, "Revenue"); ws.cell(DR, 4, "WoW%")
    for i, (_, r) in enumerate(weekly.iterrows(), 1):
        ws.cell(DR+i, 1, f"Wk{int(r['week_number'])}")
        ws.cell(DR+i, 2, round(float(r["total_revenue"]), 0))
        wow = float(r["wow_growth_pct"]) if pd.notna(r["wow_growth_pct"]) else 0
        ws.cell(DR+i, 4, round(wow, 1))

    SR = 73
    ws.cell(SR, 1, "Store"); ws.cell(SR, 2, "Revenue")
    for i, (_, r) in enumerate(kpis["store_kpi"].iterrows(), 1):
        ws.cell(SR+i, 1, r["store_name"])
        ws.cell(SR+i, 2, round(float(r["revenue"]), 0))

    CR = 81
    ws.cell(CR, 1, "Category"); ws.cell(CR, 2, "Revenue")
    for i, (_, r) in enumerate(kpis["cat_kpi"].iterrows(), 1):
        ws.cell(CR+i, 1, r["category"])
        ws.cell(CR+i, 2, round(float(r["revenue"]), 0))

    # ── Chart 1: Weekly Revenue Bar ───────────────────────────────
    c1 = BarChart()
    c1.type = "col"; c1.title = "Weekly Revenue (Rs.)"; c1.style = 10
    c1.height = 10; c1.width = 18; c1.y_axis.numFmt = "#,##0"
    d1 = Reference(ws, min_col=2, max_col=2, min_row=DR, max_row=DR+8)
    t1 = Reference(ws, min_col=1, min_row=DR+1, max_row=DR+8)
    c1.add_data(d1, titles_from_data=True); c1.set_categories(t1)
    c1.series[0].graphicalProperties.solidFill      = "0D7377"
    c1.series[0].graphicalProperties.line.solidFill = "0A5557"
    ws.add_chart(c1, "B10")

    # ── Chart 2: WoW Growth Line ──────────────────────────────────
    c2 = LineChart()
    c2.title = "Week-over-Week Growth (%)"; c2.style = 10
    c2.height = 10; c2.width = 18
    d2 = Reference(ws, min_col=4, max_col=4, min_row=DR, max_row=DR+8)
    t2 = Reference(ws, min_col=1, min_row=DR+1, max_row=DR+8)
    c2.add_data(d2, titles_from_data=True); c2.set_categories(t2)
    c2.series[0].graphicalProperties.line.solidFill = "F2A900"
    c2.series[0].graphicalProperties.line.width     = 25000
    c2.series[0].smooth = True
    ws.add_chart(c2, "H10")

    # ── Chart 3: Store Revenue Horizontal Bar ─────────────────────
    c3 = BarChart()
    c3.type = "bar"; c3.title = "Store Revenue — Wks 7 & 8"; c3.style = 10
    c3.height = 10; c3.width = 18
    d3 = Reference(ws, min_col=2, max_col=2, min_row=SR, max_row=SR+5)
    t3 = Reference(ws, min_col=1, min_row=SR+1, max_row=SR+5)
    c3.add_data(d3, titles_from_data=True); c3.set_categories(t3)
    c3.series[0].graphicalProperties.solidFill      = "1B2A4A"
    c3.series[0].graphicalProperties.line.solidFill = "0D1F33"
    ws.add_chart(c3, "B30")

    # ── Chart 4: Category Pie ─────────────────────────────────────
    c4 = PieChart()
    c4.title = "Revenue by Category"; c4.style = 10
    c4.height = 10; c4.width = 18
    d4 = Reference(ws, min_col=2, max_col=2, min_row=CR, max_row=CR+5)
    t4 = Reference(ws, min_col=1, min_row=CR+1, max_row=CR+5)
    c4.add_data(d4, titles_from_data=True); c4.set_categories(t4)
    c4.dataLabels             = DataLabelList()
    c4.dataLabels.showPercent = True
    c4.dataLabels.showCatName = True
    c4.dataLabels.showVal     = False
    c4.dataLabels.showSerName = False
    for idx, color in enumerate(["0D7377","1B2A4A","F2A900","27AE60","C0392B"]):
        pt = DataPoint(idx=idx)
        pt.graphicalProperties.solidFill = color
        c4.series[0].dPt.append(pt)
    ws.add_chart(c4, "H30")

    # ── Anomaly table row 44+ ─────────────────────────────────────
    ws.row_dimensions[51].height = 8
    ws.cell(52, 2, "ANOMALY SUMMARY")
    ws["B52"].font      = Font(bold=True, size=12, color=WHITE, name="Calibri")
    ws["B52"].fill      = fill(RED)
    ws["B52"].alignment = Alignment(horizontal="center", vertical="center")
    ws.merge_cells("B52:G52")
    ws.row_dimensions[52].height = 22

    an_cols    = [2,     3,       4,      5,        6,       7]
    an_headers = ["TYPE","STORE","WEEK","DETAIL","VALUE","VS PREV"]
    an_widths  = [16,    20,      7,     35,       14,      14]
    for col, hdr, w in zip(an_cols, an_headers, an_widths):
        c = ws.cell(53, col, hdr)
        c.font = Font(bold=True, color=WHITE, size=9, name="Calibri")
        c.fill = fill(DARK_TEXT)
        c.alignment = Alignment(horizontal="center", vertical="center")
        c.border = bdr()
        ws.column_dimensions[get_column_letter(col)].width = w
    ws.row_dimensions[53].height = 18

    row_bgs = ["FFF5F5","FFF0E0","FAFAFA","F5F0FF"]
    for i, a in enumerate(anomalies):
        r      = 54 + i
        row_bg = row_bgs[i % len(row_bgs)]
        tc     = RED if "DROP" in str(a.get("type","")) else ORANGE if "SPIKE" in str(a.get("type","")) else PURPLE
        vals   = [a.get("type",""), a.get("store",""), str(a.get("week","")),
                  a.get("detail",""), a.get("value","—"), a.get("prev_value","—")]
        for col, val in zip(an_cols, vals):
            c = ws.cell(r, col, val)
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            c.fill   = fill(row_bg)
            c.font   = Font(size=8, name="Calibri", color=tc if col==2 else DARK_TEXT, bold=(col==2))
            c.border = bdr()
        ws.row_dimensions[r].height = 18

    # ── Store scorecard row 44+ right side ───────────────────────
    ws.cell(52, 8, "STORE PERFORMANCE SCORECARD")
    ws["H52"].font      = Font(bold=True, size=12, color=WHITE, name="Calibri")
    ws["H52"].fill      = fill(NAVY)
    ws["H52"].alignment = Alignment(horizontal="center", vertical="center")
    ws.merge_cells("H52:L52")
    ws.row_dimensions[52].height = 22

    sc_cols    = [8,       9,          10,        11,      12]
    sc_headers = ["STORE","REVENUE","MARGIN%","CONV%","RETURN%"]
    for col, hdr in zip(sc_cols, sc_headers):
        c = ws.cell(53, col, hdr)
        c.font = Font(bold=True, color=WHITE, size=9, name="Calibri")
        c.fill = fill(TEAL)
        c.alignment = Alignment(horizontal="center", vertical="center")
        c.border = bdr()

    for i, (_, r) in enumerate(kpis["store_kpi"].iterrows()):
        row    = 54 + i
        row_bg = LIGHT_TEAL if i % 2 == 0 else WHITE
        rc     = GREEN if i == 0 else (RED if i == len(kpis["store_kpi"])-1 else DARK_TEXT)
        svals  = [r["store_name"], f"Rs.{r['revenue']:,.0f}",
                  f"{r['margin']:.1f}%", f"{r['conversion']:.1f}%", f"{r['return_rate']:.1f}%"]
        for col, val in zip(sc_cols, svals):
            c = ws.cell(row, col, val)
            c.alignment = Alignment(horizontal="center", vertical="center")
            c.fill   = fill(row_bg)
            c.font   = Font(size=8, name="Calibri", color=rc if col==8 else DARK_TEXT, bold=(col==8))
            c.border = bdr()
        ws.row_dimensions[row].height = 18

    # ══════════════════════════════════════════════════════════════
    # WEEKLY KPIs SHEET
    # ══════════════════════════════════════════════════════════════
    ws_kpi       = wb["wk_data"]
    ws_kpi.title = "📈 Weekly KPIs"
    ws_kpi.insert_rows(1, 3)
    ws_kpi.cell(1, 1, "RETAILPULSE AI — WEEKLY KPI TABLE")
    ws_kpi["A1"].font      = Font(bold=True, size=14, color=WHITE, name="Calibri")
    ws_kpi["A1"].fill      = fill(NAVY)
    ws_kpi["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws_kpi.merge_cells("A1:J1")
    ws_kpi.row_dimensions[1].height = 30
    for cell in ws_kpi[4]:
        if cell.value:
            cell.font = Font(bold=True, color=WHITE, size=10, name="Calibri")
            cell.fill = fill(TEAL)
            cell.alignment = Alignment(horizontal="center", wrap_text=True)
    ws_kpi.row_dimensions[4].height = 26
    for row in ws_kpi.iter_rows(min_row=5, max_row=ws_kpi.max_row):
        for cell in row:
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.fill = fill(LIGHT_BG if cell.row % 2 == 1 else WHITE)
    ws_kpi.freeze_panes = "A5"

    # ══════════════════════════════════════════════════════════════
    # STORE PERFORMANCE SHEET
    # ══════════════════════════════════════════════════════════════
    ws_st       = wb["st_data"]
    ws_st.title = "🏪 Store Performance"
    ws_st.insert_rows(1, 3)
    ws_st.cell(1, 1, "STORE PERFORMANCE — WEEKS 7 & 8")
    ws_st["A1"].font      = Font(bold=True, size=14, color=WHITE, name="Calibri")
    ws_st["A1"].fill      = fill(NAVY)
    ws_st["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws_st.merge_cells("A1:H1")
    ws_st.row_dimensions[1].height = 30
    for cell in ws_st[4]:
        if cell.value:
            cell.font = Font(bold=True, color=WHITE, size=10, name="Calibri")
            cell.fill = fill(TEAL)
            cell.alignment = Alignment(horizontal="center")
    ws_st.freeze_panes = "A5"
    if ws_st.max_row >= 5:
        for cell in ws_st[5]:
            cell.fill = fill("D5F5E3")
        for cell in ws_st[ws_st.max_row]:
            cell.fill = fill("FADBD8")

    # ══════════════════════════════════════════════════════════════
    # CATEGORY KPIs SHEET
    # ══════════════════════════════════════════════════════════════
    ws_cat       = wb["cat_data"]
    ws_cat.title = "📦 Category KPIs"
    ws_cat.insert_rows(1, 3)
    ws_cat.cell(1, 1, "CATEGORY PERFORMANCE — 8 WEEKS")
    ws_cat["A1"].font      = Font(bold=True, size=14, color=WHITE, name="Calibri")
    ws_cat["A1"].fill      = fill(NAVY)
    ws_cat["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws_cat.merge_cells("A1:F1")
    ws_cat.row_dimensions[1].height = 30
    for cell in ws_cat[4]:
        if cell.value:
            cell.font = Font(bold=True, color=WHITE, size=10, name="Calibri")
            cell.fill = fill(PURPLE)
            cell.alignment = Alignment(horizontal="center")

    # ══════════════════════════════════════════════════════════════
    # AI REPORT SHEET
    # ══════════════════════════════════════════════════════════════
    ws_ai = wb.create_sheet("🤖 AI Report")
    ws_ai.sheet_view.showGridLines = False
    ws_ai.column_dimensions["A"].width = 2
    ws_ai.column_dimensions["B"].width = 110
    ws_ai.cell(1, 2, "AI-GENERATED EXECUTIVE REPORT — RetailPulse AI")
    ws_ai["B1"].font      = Font(bold=True, size=16, color=WHITE, name="Calibri")
    ws_ai["B1"].fill      = fill(NAVY)
    ws_ai["B1"].alignment = Alignment(horizontal="center", vertical="center")
    ws_ai.merge_cells("B1:B2")
    ws_ai.row_dimensions[1].height = 28
    ws_ai.row_dimensions[2].height = 18
    ws_ai.cell(3, 2, f"Generated by Groq AI  ·  {datetime.now().strftime('%d %B %Y, %H:%M')}  ·  Replacing 4 hours of manual work")
    ws_ai["B3"].font      = Font(italic=True, size=10, color="666666", name="Calibri")
    ws_ai["B3"].fill      = fill(LIGHT_BG)
    ws_ai["B3"].alignment = Alignment(horizontal="center")
    ws_ai.row_dimensions[3].height = 18
    ws_ai.row_dimensions[4].height = 8
    r = 5
    for line in ai_report.split("\n"):
        c = ws_ai.cell(r, 2, line)
        c.alignment = Alignment(wrap_text=True, vertical="top")
        stripped = line.strip()
        if stripped.startswith("**") or stripped[:2] in ["1.","2.","3.","4.","5.","6."]:
            c.font = Font(bold=True, size=13, color=NAVY, name="Calibri")
            c.fill = fill(LIGHT_TEAL)
            ws_ai.row_dimensions[r].height = 24
        elif stripped.startswith(("*","-","•")):
            c.font = Font(size=11, color=DARK_TEXT, name="Calibri")
            c.fill = fill(LIGHT_BG)
            ws_ai.row_dimensions[r].height = 28
        elif not stripped:
            ws_ai.row_dimensions[r].height = 8
        else:
            c.font = Font(size=11, color=DARK_TEXT, name="Calibri")
            ws_ai.row_dimensions[r].height = 32
        r += 1

    # ══════════════════════════════════════════════════════════════
    # ANOMALIES SHEET
    # ══════════════════════════════════════════════════════════════
    if "Anomalies" in wb.sheetnames:
        ws_an = wb["Anomalies"]
        ws_an.insert_rows(1, 3)
        ws_an.cell(1, 1, "ANOMALY DETECTION REPORT — Auto-Generated by RetailPulse AI")
        ws_an["A1"].font      = Font(bold=True, size=13, color=WHITE, name="Calibri")
        ws_an["A1"].fill      = fill(RED)
        ws_an["A1"].alignment = Alignment(horizontal="center", vertical="center")
        ws_an.merge_cells("A1:G1")
        ws_an.row_dimensions[1].height = 30
        for cell in ws_an[4]:
            if cell.value:
                cell.font      = Font(bold=True, color=WHITE, name="Calibri")
                cell.fill      = fill(DARK_TEXT)
                cell.alignment = Alignment(horizontal="center")

    # ══════════════════════════════════════════════════════════════
    # RAW DATA SHEET
    # ══════════════════════════════════════════════════════════════
    ws_raw = wb["Raw Data"]
    ws_raw.insert_rows(1, 2)
    ws_raw.cell(1, 1, "RetailPulse AI — Transaction Data")
    ws_raw["A1"].font  = Font(bold=True, size=14, color=NAVY, name="Calibri")
    ws_raw.cell(2, 1, f"Generated: {datetime.now().strftime('%d %b %Y %H:%M')} | Rows: {len(df):,}")
    ws_raw["A2"].font  = Font(italic=True, size=9, color="666666", name="Calibri")
    for cell in ws_raw[3]:
        cell.font      = Font(bold=True, color=WHITE, name="Calibri", size=10)
        cell.fill      = fill(NAVY)
        cell.alignment = Alignment(horizontal="center")
    ws_raw.freeze_panes = "A4"
    for col in ws_raw.columns:
        max_len = max((len(str(c.value or "")) for c in col), default=8)
        ws_raw.column_dimensions[get_column_letter(col[0].column)].width = min(max_len+2, 22)

    # ── Reorder sheets ────────────────────────────────────────────
    order = ["📊 Dashboard","🤖 AI Report","📈 Weekly KPIs",
             "🏪 Store Performance","📦 Category KPIs","Anomalies","Raw Data"]
    for name in order:
        if name in wb.sheetnames:
            wb.move_sheet(name, offset=-(wb.sheetnames.index(name)))

    wb.save(filename)
    print(f"   ✅ Excel Dashboard saved!")
    print(f"   📂 File: {filename}")
    print(f"   📊 7 Sheets: Dashboard + AI Report + Weekly KPIs + Store + Category + Anomalies + Raw Data\n")

    from google.colab import files
    files.download(filename)
    print(f"   ⬇️  Download started!\n")
    return filename

In [8]:
def run_retailpulse_agent():
    """
    The main agent function. Runs all 5 steps end-to-end.

    AGENT PIPELINE:
    Data → KPIs → Anomalies → AI Report → Excel Export

    This is what "agentic AI" means:
    Multiple tools working together, orchestrated by a central controller,
    with AI making intelligent decisions along the way.
    """
    print("\n" + "═" * 65)
    print("  🛒  RetailPulse AI — Business Intelligence Agent  🤖")
    print("  Class 12 | End-to-End Portfolio Project")
    print("═" * 65 + "\n")

    start = datetime.now()

    # --- Run the pipeline ---
    df        = generate_retail_data()
    kpis      = compute_kpis(df)
    anomalies = detect_anomalies(df, kpis)
    ai_report = generate_ai_report(kpis, anomalies, df)
    file_path = export_to_excel(df, kpis, anomalies, ai_report)

    elapsed = (datetime.now() - start).seconds

    print("═" * 65)
    print("  ✅ RETAILPULSE AI AGENT — COMPLETED SUCCESSFULLY!")
    print("═" * 65)
    print(f"\n  ⏱  Total Runtime: {elapsed} seconds")
    print(f"  📊  KPI Tables:   4 sheets computed automatically")
    print(f"  🚨  Anomalies:    {len(anomalies)} issues flagged")
    print(f"  🤖  AI Report:    Executive-level report generated")
    print(f"  📁  Output:       {file_path}")
    print(f"\n  💼  Manual work replaced: ~4 hours every Monday morning")
    print(f"  🚀  Add this to your resume & portfolio!\n")

    return file_path



In [9]:
import os
os.makedirs("/content/retailpulse_outputs", exist_ok=True)

if __name__ == "__main__":
    output_file = run_retailpulse_agent()


═════════════════════════════════════════════════════════════════
  🛒  RetailPulse AI — Business Intelligence Agent  🤖
  Class 12 | End-to-End Portfolio Project
═════════════════════════════════════════════════════════════════

📦 STEP 1: Generating retail dataset...
   (In real jobs: this is your SQL/CSV import)

   ✅ Dataset created: 1,400 rows × 18 columns
   📅 Date range: 2024-11-04 to 2024-12-29
   🏪 Stores: 5 | Categories: 5

📊 STEP 2: Computing KPIs (replacing Monday morning Excel work)...

   📈 Weekly Revenue Summary:
      Week 1: ₹  13,129,680  |  WoW:   —  
      Week 2: ₹  13,111,155  |  WoW: ▼ 0.1%
      Week 3: ₹  13,088,312  |  WoW: ▼ 0.2%
      Week 4: ₹  13,038,565  |  WoW: ▼ 0.4%
      Week 5: ₹  13,063,721  |  WoW: ▲ 0.2%
      Week 6: ₹  11,535,836  |  WoW: ▼ 11.7%
      Week 7: ₹  13,185,445  |  WoW: ▲ 14.3%
      Week 8: ₹  15,876,876  |  WoW: ▲ 20.4%

   🏪 Store Performance (Weeks 7-8):
      Mumbai Central         | Rev: ₹10,051,689 | Margin: 38.4%
      Delhi C

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   ⬇️  Download started!

═════════════════════════════════════════════════════════════════
  ✅ RETAILPULSE AI AGENT — COMPLETED SUCCESSFULLY!
═════════════════════════════════════════════════════════════════

  ⏱  Total Runtime: 3 seconds
  📊  KPI Tables:   4 sheets computed automatically
  🚨  Anomalies:    4 issues flagged
  🤖  AI Report:    Executive-level report generated
  📁  Output:       /content/retailpulse_outputs/RetailPulse_20260609_0636.xlsx

  💼  Manual work replaced: ~4 hours every Monday morning
  🚀  Add this to your resume & portfolio!

